In [ ]:
import os,sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.append('/root/liubo/TravDiT')  # 添加项目根目录到 Python 路径
from args import make_args
from datasets import Dataset
from util import load_raw_data,load_vec,generate_trajectory_prompt_temp,temporal_pattern_one_hot_batch
import random
from datetime import timedelta
import pandas as pd
import torch

args = make_args()
unique_poi_types = args.unique_poi_types
random.seed(args.seed)  # 设置随机种子以确保可重复性

In [ ]:
all_texts, all_labels = [], []
dow_map = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
city_list = ['Changsha', 'Guangzhou', 'Shenzhen']
sample_size = 8000 # for one city

for city_id, city in enumerate(city_list):
    raw_data = load_raw_data(city=city)
    vec = load_vec(city=city)

    indices = random.sample(range(len(raw_data)), sample_size)
    sampled_raw_data = [raw_data[i] for i in indices]

    texts, labels = [], []
    for item in sampled_raw_data:
        traj = item["traj_region_id"]
        start_time = pd.to_datetime(item['tod'])  # 起始时间点
        i = 0
        while i < len(traj) - 1:
            start_region_id = traj[i]
            # === 计算在当前起点的停留时长 ===
            stay_slots = 1
            j = i + 1
            while j < len(traj) and traj[j] == start_region_id:
                stay_slots += 1
                j += 1
            stay_duration_hours = stay_slots * 0.5
            # 出发时间 = 离开该点的时刻
            departure_time = start_time + timedelta(minutes=30 * (i + stay_slots - 1))
            tod = departure_time.strftime("%H:%M")
            dow = dow_map[departure_time.dayofweek]
            # prompt 构造
            departure_poi_dist = vec[int(start_region_id)][2: 2 + args.poi_dim]
            prompt = generate_trajectory_prompt_temp(
                city=city,
                tod=tod,
                dow=dow,
                departure_poi_dist=departure_poi_dist,
                poi_category_list=args.unique_poi_types
            )
            texts.append(prompt)
            # 目的地 POI（第一个不同于起点的位置）
            if j < len(traj):
                dest_region_id = traj[j]
                dest_poi_dist = vec[int(dest_region_id)][2: 2 + args.poi_dim]
            else:
                dest_poi_dist = torch.zeros(args.poi_dim)
            label = [city_id] + [stay_duration_hours] + dest_poi_dist.tolist()
            labels.append(label)
            i = j  # 下一个起点

    all_texts.extend(texts)
    all_labels.extend(labels)

# ========== 构造混合 Dataset ==========
dataset = Dataset.from_dict({
    "text": all_texts,
    "label": all_labels
})

# ========== 打乱并划分 ==========
dataset = dataset.shuffle(seed=args.seed)
split_idx = int(0.8 * len(dataset))
train_dataset = dataset.select(range(split_idx))
valid_dataset = dataset.select(range(split_idx, len(dataset)))

In [ ]:
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          AutoModel,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
)
from trainer.customTrainer import CustomTrainer
base_model  = "/datadisk/llama2"
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True,cache_dir = "./models/")
tokenizer.pad_token = tokenizer.eos_token
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=512)

traj_train_df = train_dataset.map(preprocess_function, batched=True)
traj_valid_df = valid_dataset.map(preprocess_function, batched=True)

In [ ]:
from model.model_LLM import temporalTripModel
model = temporalTripModel(base_model_path="/datadisk/llama2")

In [ ]:
# # Configure training parameters
training_params = TrainingArguments(
    output_dir="/checkpt",
    num_train_epochs=args.llm_epoch,
    per_device_train_batch_size=args.llm_batch_size,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_strategy='no',
    save_total_limit=0,
    load_best_model_at_end=False,
    evaluation_strategy='epoch',
    logging_steps=32,
    learning_rate=5e-5,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine"
)


# Initialize and configure the trainer
trainer = CustomTrainer(
    model= model,
    train_dataset= traj_train_df,
    eval_dataset= traj_valid_df,
    tokenizer= tokenizer,
    args= training_params,
    compute_metrics= None,
)


In [ ]:
trainer.train()

In [ ]:
# Save the fine-tuned model and tokenizer
trainer.model.save_pretrained(save_path=f"/root/liubo/TravDiT/{args.checkpt_path}/pretrain/llm")
trainer.tokenizer.save_pretrained(save_directory=f"/root/liubo/TravDiT/{args.checkpt_path}/pretrain/llm")
torch.save(model.trajectory_head.state_dict(), f"/root/liubo/TravDiT/{args.checkpt_path}/pretrain/llm/trajectory_head.pt")